In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

# Calculate C2SM

In [ ]:
def calculate_C2SM(mass_raw, dt, T):

    a1, A11, E1, a2, A12, E2 = 0.7, 2e7, 8.054e7, 1, 1.3e7, 1.672e8

    RR = 8314.462

    kappa1 = A11 * np.exp(-E1 / (RR * T))
    kappa2 = A12 * np.exp(-E2 / (RR * T))

    dMassDV = np.minimum( dt * mass_raw * (kappa1 * a1 + kappa2 * a2), (kappa1 * a1 + kappa2 * a2) / (kappa1 + kappa2 + 1e-12) * mass_raw)
    dMassC = np.minimum(dt * mass_raw * (kappa1 * (1 - a1) + kappa2 * (1 - a2)), mass_raw - dMassDV)

    return dMassDV, dMassC

In [ ]:
Tmax = 1400
tTmax = 1e-1

t0 = 0
massDV = [0]
massC = [0]
mass_raw = [1]
mass = [1]

ts = np.arange(0,0.06,5e-6)

Ts = np.ones(len(ts)) * 800

for t, T  in zip(ts[1:],Ts):
    dt = t - t0
    dMassDV, dMassC = calculate_C2SM(mass_raw[-1],dt,T)
    mass_raw.append(mass_raw[-1] - dMassDV - dMassC)
    massDV.append(massDV[-1] + dMassDV)
    massC.append(massC[-1] + dMassC)
    mass.append(massC[-1] + mass_raw[-1])
    t0 = t


In [ ]:

fig, ax = plt.subplots(1,1)

ax.plot(ts,mass_raw,label='Raw solid fuel')
ax.plot(ts,massDV,label='Volatiles')
ax.plot(ts,massC,label='Char')
ax.plot(ts,mass,label='Mass/Mass0')
ax.plot(ts,np.array(massC)/np.array(mass),label='Ychar')
ax.set_ylabel('Y (-)')
ax.set_xlabel('t (s)')
ax.legend()
ax2 = ax.twinx()
ax2.plot(ts,Ts,linestyle='--')
ax2.set_ylabel('T (K)')

In [ ]:
fig.savefig('reference/figures/C2SM_python.png')

In [ ]:
print('Final mass volatiles:',massDV[-1]/mass_raw[0])
print('Final mass/initial mass:',mass[-1])
print('Final char mass fraction',massC[-1]/mass[-1])


## Read particles

Verify particle mass profile

In [ ]:
import json
import os 
import pyvista as pv

prt_dfs = {}
for cloud_path, cloud_name in zip(['writeCarbCloud'],['carbCloud']):
    path = os.path.join('foam_template/postProcessing', cloud_path)
    with open(os.path.join(path,cloud_name + '.vtp.series')) as f:
        d = json.load(f)

    time_file_dict = {}
    times = []
    files = []
    for di in d['files']:
        times.append(di['time'])
        files.append(os.path.join(path,di['name']))

    sorting = np.argsort(np.array(times))
    files = list(np.array(files)[sorting])

    print('Reading',cloud_name)
    data_list = []
    for file in files:
        vtk = pv.read(file)
        data_list.append(np.concatenate((vtk['T'][:,np.newaxis],vtk['origId'][:,np.newaxis],vtk['d'][:,np.newaxis],vtk['YC(s)'][:,np.newaxis],vtk['TimeValue']*np.ones(vtk.n_points)[:,np.newaxis]),axis=1))
    data = np.concatenate(tuple(data_list))
    prt_dfs[cloud_name] = pd.DataFrame({'T':data[:,0],'origId':data[:,1],'d':data[:,2],'YC(s)':data[:,3],'Time':data[:,4]})

In [ ]:
carb_df = prt_dfs['carbCloud']

In [ ]:
rho_prt = 707
mass0 = 1e-11
carb_df['mass'] = carb_df['d']**3/6*np.pi*rho_prt

fig, ax = plt.subplots(1,1)

ax.plot(carb_df['Time'],carb_df['mass']/mass0,label='mass/mass0')
ax.plot(carb_df['Time'],carb_df['YC(s)'],label='Ychar')
ax.set_ylabel('Y (-)')
ax.legend()
ax.set_xlabel('t (s)')
ax2 = ax.twinx()
ax2.plot(carb_df['Time'],carb_df['T'],linestyle='--')
ax2.set_ylabel('T (K)')

In [ ]:
fig.savefig('reference/figures/C2SM_openfoam.png')

# Read outlet

In [ ]:
df = pd.read_csv('temp/test_C2SM/postProcessing/sumZ/0/surfaceFieldValue.dat',header=5,sep=r'\t',engine='python')

In [ ]:
df

In [ ]:
m_volatiles = np.sum(df['weightedSum(Z)']*5e-6)
print(m_volatiles)

# Comparison

In [ ]:
carb_df['mass/mass0'] = carb_df['d']**3/6*np.pi*rho_prt/mass0
YCs = np.array(massC) / np.array(mass)


fig, ax = plt.subplots(1,1)
ax.plot(carb_df['Time'],carb_df['mass/mass0'],label='mass/mass0 OpenFOAM')
ax.plot(ts,mass,label='Mass/Mass0 Python',linestyle='--')

ax.plot(carb_df['Time'],carb_df['YC(s)'],label='Ychar OpenFOAM')
ax.plot(ts,YCs,label='YChar Python',linestyle='--')

ax.legend()
ax.set_ylabel('Y (-)')
ax.set_xlabel('t (s)')

In [ ]:
fig.savefig('reference/figures/comparison.png')

In [ ]:
vol_rel = np.array(m_volatiles) / (np.array(massDV[-1])*1e-10) # 1e-10 is injected mass
vol_rel

In [ ]:
mass_rel = carb_df['mass'].iloc[-1] / (mass[-1]*1e-10/10) # 1e-10 is injected mass, 10 particles per parcel
mass_rel

## Write reference data

In [ ]:
np.allclose(carb_df['mass/mass0'],mass[::20],atol=1.1e-2)

In [ ]:
np.allclose(carb_df['YC(s)'],YCs[::20],atol=0.016)

In [ ]:
np.allclose(np.array(m_volatiles), (np.array(massDV[-1])*1e-10))

In [ ]:
df_profiles = pd.DataFrame({'Time':ts[::20],'mass/mass0':mass[::20],'YC(s)':YCs[::20]})
df_profiles.to_csv('reference/foam/profiles.csv')

In [ ]:
df_final = pd.DataFrame({'mass_final_particle':[mass[-1]*1e-10/10], 'mass_volatiles_final':[massDV[-1]*1e-10]})
# mass of one particle, volatiles of 10 particles
df_final.to_csv('reference/foam/final.csv')

In [ ]:

final_reference_df = pd.read_csv('reference/foam/profiles.csv',index_col=0)

In [ ]:
final_reference_df